In [1]:
import os
import numpy as np
import pandas as pd

import cv2
import matplotlib.pyplot as plt
import seaborn as sns

# Use tf_keras for compatibility with this environment (Keras 3 breaks legacy APIs here)
import tf_keras as keras
from tf_keras import layers, models, optimizers
from tf_keras.preprocessing.image import ImageDataGenerator

from tqdm import tqdm

# Reproducibility (score-neutral; helps stability)
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
try:
    keras.utils.set_random_seed(SEED)
except Exception:
    pass

# Resolve correct Kaggle input paths robustly
BASE_CANDIDATES = [
    "/kaggle/input/aerial-cactus-identification",
    "../input/aerial-cactus-identification",
    "/kaggle/data/input/aerial-cactus-identification",
    "../input",
]
BASE = None
for c in BASE_CANDIDATES:
    if os.path.exists(os.path.join(c, "train.csv")) and os.path.isdir(
        os.path.join(c, "train")
    ):
        BASE = c
        break
if BASE is None:
    raise FileNotFoundError(
        "Could not locate aerial-cactus-identification dataset folder under expected paths."
    )

train_dir = os.path.join(BASE, "train")
test_dir = os.path.join(BASE, "test")
train_csv_path = os.path.join(BASE, "train.csv")
sample_sub_path = os.path.join(BASE, "sample_submission.csv")

print("BASE:", BASE)
print(
    "train_dir exists:", os.path.isdir(train_dir), "files:", len(os.listdir(train_dir))
)
print("test_dir exists:", os.path.isdir(test_dir), "files:", len(os.listdir(test_dir)))
print("train.csv exists:", os.path.exists(train_csv_path))
print("sample_submission.csv exists:", os.path.exists(sample_sub_path))



2026-01-11 00:18:56.911355: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768090736.925433      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768090736.929645      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-11 00:18:56.947920: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

BASE: /kaggle/input/aerial-cactus-identification
train_dir exists: True files: 14176
test_dir exists: True files: 3326
train.csv exists: True
sample_submission.csv exists: True


In [2]:
train = pd.read_csv(train_csv_path)
df_test = pd.read_csv(sample_sub_path)

# Ensure expected columns exist
assert set(["id", "has_cactus"]).issubset(train.columns)
assert set(["id", "has_cactus"]).issubset(df_test.columns)

train.head(5)



,id,has_cactus
0,2de8f189f1dce439766637e75df0ee27.jpg,1
1,36704d250f236238e7f996812c48235d.jpg,1
2,eacde22fdc8c175972a5768e3daa8bc9.jpg,1
3,5d442f834da5e57d22b24802c32a8ca8.jpg,1
4,152491e0daf75c0e669400300ff7e645.jpg,1


In [3]:
# Keras flow_from_dataframe with class_mode='binary' expects labels as strings for two-class mapping in many setups
train["has_cactus"] = train["has_cactus"].astype(str)

print(train.shape[0], train.shape[1])
print(train["has_cactus"].value_counts())



14175 2
has_cactus
1    10628
0     3547
Name: count, dtype: int64


In [4]:
# Display an example image if running in notebook; safe in script too.
example_path = os.path.join(train_dir, train.iloc[1, 0])
print("Example image path:", example_path, "exists:", os.path.exists(example_path))



Example image path: /kaggle/input/aerial-cactus-identification/train/36704d250f236238e7f996812c48235d.jpg exists: True


In [5]:
# Data generators (same core idea; just ensures the object exists in this environment)
datagen = ImageDataGenerator(rescale=1.0 / 255.0)
batch_size = 150

# Keep the original split logic (minimal change), but avoid overlap: [0:15000] and [15000:]
train_df = train.iloc[:15000].copy()
val_df = train.iloc[15000:].copy()

train_generator = datagen.flow_from_dataframe(
    dataframe=train_df,
    directory=train_dir,
    x_col="id",
    y_col="has_cactus",
    class_mode="binary",
    batch_size=batch_size,
    target_size=(150, 150),
    shuffle=True,
    seed=SEED,
)

validation_generator = datagen.flow_from_dataframe(
    dataframe=val_df,
    directory=train_dir,
    x_col="id",
    y_col="has_cactus",
    class_mode="binary",
    batch_size=50,
    target_size=(150, 150),
    shuffle=False,
    seed=SEED,
)



Found 14175 validated image filenames belonging to 2 classes.


ValueError: If class_mode="binary" there must be 2 classes. Found 0 classes.

In [6]:
# Model definition preserved (same layers/order/activations)
model = models.Sequential()
model.add(layers.Conv2D(32, (3, 3), activation="relu", input_shape=(150, 150, 3)))
model.add(layers.MaxPool2D((2, 2)))
model.add(layers.Conv2D(64, (3, 3), activation="relu", input_shape=(150, 150, 3)))
model.add(layers.MaxPool2D((2, 2)))
model.add(layers.Conv2D(128, (3, 3), activation="relu", input_shape=(150, 150, 3)))
model.add(layers.MaxPool2D((2, 2)))
model.add(layers.Conv2D(128, (3, 3), activation="relu", input_shape=(150, 150, 3)))
model.add(layers.MaxPool2D((2, 2)))
model.add(layers.Flatten())
model.add(layers.Dense(512, activation="relu"))
model.add(layers.Dense(256, activation="relu"))
model.add(layers.Dense(128, activation="relu"))
model.add(layers.Dense(1, activation="sigmoid"))

model.summary()



Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 148, 148, 32)      896       
                                                                 
 max_pooling2d (MaxPooling2  (None, 74, 74, 32)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 72, 72, 64)        18496     
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 36, 36, 64)        0         
 g2D)                                                            
                                                                 
 conv2d_2 (Conv2D)           (None, 34, 34, 128)       73856     
                                                                 
 max_pooling2d_2 (MaxPoolin  (None, 17, 17, 128)       0

2026-01-11 00:19:04.140973: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1768090744.141460      11 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:46:00.0, compute capability: 8.6


In [7]:
# Fix optimizer API: use RMSprop class (equivalent intent to old optimizers.rmsprop())
model.compile(
    loss="binary_crossentropy", optimizer=optimizers.RMSprop(), metrics=["accuracy"]
)



In [8]:
# Fix deprecated fit_generator: use fit() (same training semantics)
epochs = 10
history = model.fit(
    train_generator,
    steps_per_epoch=100,
    epochs=epochs,
    validation_data=validation_generator,
    validation_steps=50,
    verbose=1,
)



NameError: name 'validation_generator' is not defined

In [9]:
# Plotting with correct metric keys across Keras versions
hist = history.history
acc_key = "accuracy" if "accuracy" in hist else ("acc" if "acc" in hist else None)
val_acc_key = (
    "val_accuracy"
    if "val_accuracy" in hist
    else ("val_acc" if "val_acc" in hist else None)
)

if acc_key is not None and val_acc_key is not None:
    epochs_ = range(0, epochs)
    plt.figure(figsize=(6, 4))
    plt.plot(epochs_, hist[acc_key], label="training accuracy")
    plt.scatter(epochs_, hist[val_acc_key], label="validation accuracy")
    plt.xlabel("no of epochs")
    plt.ylabel("accuracy")
    plt.title("no of epochs vs accuracy")
    plt.legend()
    plt.show()

plt.figure(figsize=(6, 4))
plt.plot(range(0, epochs), hist["loss"], label="training loss")
plt.scatter(range(0, epochs), hist["val_loss"], label="validation loss")
plt.xlabel("No of epochs")
plt.ylabel("loss")
plt.title("no of epochs vs loss")
plt.legend()
plt.show()



NameError: name 'history' is not defined

In [10]:
# Load test images in the SAME order as sample_submission ids to guarantee alignment.
# Also ensure cv2 reads correctly and converts BGR->RGB consistently (minor, but avoids channel mismatch issues).
test_ids = df_test["id"].tolist()

test_features = np.zeros((len(test_ids), 150, 150, 3), dtype=np.float32)
missing = 0

for i, img_id in enumerate(tqdm(test_ids, desc="Loading test images")):
    img_path = os.path.join(test_dir, img_id)
    img = cv2.imread(img_path)
    if img is None:
        missing += 1
        # Fallback to zeros; should not happen in normal Kaggle environment
        continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (150, 150), interpolation=cv2.INTER_AREA)
    test_features[i] = img.astype(np.float32) / 255.0

print("Missing test images:", missing)
print("test_features shape:", test_features.shape)



Loading test images: 100%|██████████| 3325/3325 [00:01<00:00, 2853.52it/s]

Missing test images: 0
test_features shape: (3325, 150, 150, 3)


In [11]:
# Predict probabilities (do NOT threshold; ROC AUC expects probabilities and submission requires a probability)
test_predictions = model.predict(test_features, batch_size=64, verbose=1).reshape(-1)

# Build submission aligned to sample_submission ids
submissions = pd.DataFrame(
    {"id": test_ids, "has_cactus": test_predictions.astype(float)}
)

# Sanity checks
assert submissions.shape[0] == df_test.shape[0]
assert submissions["id"].iloc[0] == df_test["id"].iloc[0]

submissions.head()



I0000 00:00:1768090747.079626      63 cuda_dnn.cc:529] Loaded cuDNN version 90300


52/52 [==============================] - 1s 8ms/step


,id,has_cactus
0,09034a34de0e2015a8a28dfe18f423f6.jpg,0.501678
1,134f04305c795d6d202502c2ce3578f3.jpg,0.503174
2,41fad8d145e6c41868ce3617e30a2545.jpg,0.502430
3,35f8a11352c8d41b6231bb33d8d09f7e.jpg,0.502144
4,b77dc902b035887cbbc01920ce0e3151.jpg,0.501366


In [12]:
# Write valid submission file
out_path = "submission.csv"
submissions.to_csv(out_path, index=False)
print("Wrote:", out_path, "rows:", len(submissions), "cols:", list(submissions.columns))
print(submissions.describe(include="all"))

Wrote: submission.csv rows: 3325 cols: ['id', 'has_cactus']
                                          id   has_cactus
count                                   3325  3325.000000
unique                                  3325          NaN
top     09034a34de0e2015a8a28dfe18f423f6.jpg          NaN
freq                                       1          NaN
mean                                     NaN     0.502378
std                                      NaN     0.000833
min                                      NaN     0.500341
25%                                      NaN     0.501788
50%                                      NaN     0.502278
75%                                      NaN     0.502878
max                                      NaN     0.506269


/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
